In [15]:
import typesense
import pandas as pd
import sqlite3
from datetime import datetime, timezone
from IPython.display import display, HTML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm
model = SentenceTransformer('all-MiniLM-L6-v2')

import numpy as np

import time
import json
import re

db_path = '/Users/zphilipp/git/research/dealsdb/deals_db1.db'
index_name = 'deals' 
    
words_to_remove = [
    "about", "above", "across", "after", "against", "along", "among", "around", 
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "toward", "under",
    "until", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a", "b", "c", "d", "e", "f", "g", "h",
    "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u",
    "v", "w", "x", "y", "z", "aa", "aaa", "aaaa", #, "to", "up", "with", "at",
]

words_pattern = r'\b(' + '|'.join(map(re.escape, words_to_remove)) + r')\b'

def remove_prepositions_and_conjunctions(text):
    text = remove_chinese_characters(text).lower()
    cleaned_text = re.sub(words_pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"[();,'\\\[\]0-9]+", "", cleaned_text) # special charch and one place numbers
    return re.sub(r'\s+', ' ', cleaned_text).strip()

def remove_chinese_characters(text):
    # Regular expression pattern to match Chinese characters
    patt = r'[\u4e00-\u9fff]+'
    # Substitute Chinese characters with an empty string
    cleaned_text = re.sub(patt, '', text)
    return cleaned_text
    
#sql_query = """
#        SELECT                                 
#            d.id,
#            
#            COALESCE(MAX(d.title_general), '') || ' ' ||
#            COALESCE(GROUP_CONCAT(o.title, ','), '') || ' ' ||
#            COALESCE(MAX(m.name), '') AS text,
#            
#            d.customer_category_id
#        FROM deals d
#            LEFT JOIN merchant m ON (d.merchant_id=m.id)
#            LEFT JOIN options o ON (o.deal_id=d.id)
#        -- where d.id=151
#        GROUP BY d.id
#"""

def remove_prepositions_and_conjunctions(text):
    text = remove_chinese_characters(text).lower()
    cleaned_text = re.sub(words_pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"[();,'\\\[\]0-9]+", "", cleaned_text) # special charch and one place numbers
    return re.sub(r'\s+', ' ', cleaned_text).strip()

def remove_chinese_characters(text):
    # Regular expression pattern to match Chinese characters
    patt = r'[\u4e00-\u9fff]+'
    # Substitute Chinese characters with an empty string
    cleaned_text = re.sub(patt, '', text)
    return cleaned_text

#def get_important_words(document):
    #document = remove_prepositions_and_conjunctions(document)
    #document = [document]

    #vectorizer = TfidfVectorizer(ngram_range=(1, 3))
    #tfidf_matrix = vectorizer.fit_transform(document)

    #feature_names = vectorizer.get_feature_names_out()
    #first_document_vector = tfidf_matrix[0]
    
#    important_words = sorted(
#        [(feature_names[i], first_document_vector[0, i]) for i in first_document_vector.nonzero()[1]],
#        key=lambda x: x[1], reverse=True
#    )
#    return important_words

def to_dict(important_words):
    result_dict = {
        'unigram': [],
        'bigram': [],
        'trigram': []
    }
    for word, weight in important_words:
        if weight < 0.03:
            continue
        #print (word, weight)
        word = word.encode().decode('unicode_escape')
        word_count = len(word.split())

        if word_count == 1:
            result_dict['unigram'].append((word, weight))
        elif word_count == 2:
            result_dict['bigram'].append((word, weight))
        elif word_count == 3:
            result_dict['trigram'].append((word, weight))
    return json.dumps(result_dict)

In [16]:
client = typesense.Client({
    'api_key': 'MojeTajneHeslo12',
    'nodes': [{
        'host': 'localhost',  # nebo adresa vašeho serveru
        'port': '8108',      # port, na kterém běží Typesense
        'protocol': 'http'   # nebo 'https', pokud používáte SSL
    }],
    'connection_timeout_seconds': 2
})

In [17]:
client.collections["deals"].delete()

{'created_at': 1748244678,
 'default_sorting_field': '',
 'enable_nested_fields': False,
 'fields': [{'facet': False,
   'index': True,
   'infix': False,
   'locale': '',
   'name': 'deal_uuid',
   'optional': False,
   'sort': False,
   'stem': False,
   'stem_dictionary': '',
   'store': True,
   'type': 'string'},
  {'facet': False,
   'index': True,
   'infix': False,
   'locale': '',
   'name': 'document',
   'optional': False,
   'sort': False,
   'stem': False,
   'stem_dictionary': '',
   'store': True,
   'type': 'string'},
  {'facet': False,
   'hnsw_params': {'M': 16, 'ef_construction': 200},
   'index': True,
   'infix': False,
   'locale': '',
   'name': 'embedding',
   'num_dim': 384,
   'optional': False,
   'sort': False,
   'stem': False,
   'stem_dictionary': '',
   'store': True,
   'type': 'float[]',
   'vec_dist': 'cosine'},
  {'facet': False,
   'index': True,
   'infix': False,
   'locale': '',
   'name': 'category',
   'optional': False,
   'sort': False,
   's

In [18]:
schema = {
    'name': 'deals',  # název kolekce
    'fields': [
        {'name': 'deal_uuid', 'type': 'string' },
        {'name': 'document', 'type': 'string' },
        {'name': 'embedding', 'type': 'float[]', 'num_dim': 384},
        {'name': 'category', 'type': 'string' },
        {'name': 'timestamp', 'type': 'int32' },
    ],
    #'default_sorting_field': 'year'
}

# vytvoření kolekce
try:
    response = client.collections.create(schema)
except Exception as e:
    print("Kolekce pravděpodobně již existuje nebo došlo k chybě:", e)

In [19]:
def fetch_tree(item_id, cursor):
    """Funkce pro načtení stromu pro daný item_id."""
    # Načtěte vlastnosti aktuální položky
    cursor.execute("SELECT parent_id, name FROM category WHERE id = ?", (item_id,))
    item = cursor.fetchone()
    #print(item)
    if not item:
        return None
    
    # Vytvoření uzlu
    return {
        'value': item[1],
        'parent_id': item[0]
    }

def get_taxonomy_path(id, cursor):

    data = fetch_tree(id, cursor)
    category_string = ""
    while data is not None:    
        data = fetch_tree(data["parent_id"], cursor)
        try:
            category_string = data["value"] + " / " + category_string
        except TypeError as e:
            #print (data)
            break
    
    return category_string[:-2]  


In [22]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
sql_query = """
        SELECT                                 
            d.deal_uuid,
            
            COALESCE(MAX(d.title_general), '') || '. ' ||
            COALESCE(MAX(d.highlights), '') || '. ' ||
            COALESCE(GROUP_CONCAT(o.title, ','), '') || '. ' ||
            COALESCE(MAX(m.name), '') AS text,
            
            d.customer_category_id
        FROM deals d
            LEFT JOIN merchant m ON (d.merchant_id=m.id)
            LEFT JOIN options o ON (o.deal_id=d.id)
        -- where d.id=151
        GROUP BY d.deal_uuid
    """
cursor.execute(sql_query)

results = []

rows = cursor.fetchall()
with tqdm(total=len(rows)) as pbar:
    for row in rows:

        # index when deal have category only
        if row[2]:
            try:

                #print (f"Row 1: {row[1]}")
                category = get_taxonomy_path(row[2], cursor)
                document = row[1].encode().decode('unicode_escape')
                #document = remove_prepositions_and_conjunctions(document + '. ' + category.strip() + '.')


                results.append({
                    'deal_uuid': row[0],
                    'document': document,
                    'category': category,
                    'timestamp': int(datetime.now(timezone.utc).timestamp())
                })
            except Exception as e:
                print (f"Exception.: {e} Row 1 {row[1]}.")
                print (f"Error in document id.: {row[0]}.")
                
                errors = errors + 1
                raise
        pbar.update(1)
    
conn.close()
df = pd.DataFrame(results)
output_path = "deals.csv"
df.to_csv(output_path, index=False)
print(f"✅ Výsledky uloženy do: {output_path}")

  0%|          | 0/105789 [00:00<?, ?it/s]

/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_18004/1844204885.py:33: DeprecationWarning: invalid escape sequence '\S'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_18004/1844204885.py:33: DeprecationWarning: invalid escape sequence '\A'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_18004/1844204885.py:33: DeprecationWarning: invalid escape sequence '\o'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_18004/1844204885.py:33: DeprecationWarning: invalid escape sequence '\ '
  document = row[1].encode().decode('unicode_escape')


✅ Výsledky uloženy do: deals.csv


In [35]:
counter  = 0
counterTotal = 0
errors = 0

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
sql_query = """
        SELECT                                 
            d.deal_uuid,
            
            COALESCE(MAX(d.title_general), '') || '. ' ||
            COALESCE(MAX(d.highlights), '') || '. ' ||
            COALESCE(GROUP_CONCAT(o.title, ','), '') || '. ' ||
            COALESCE(MAX(m.name), '') AS text,
            
            d.customer_category_id
        FROM deals d
            LEFT JOIN merchant m ON (d.merchant_id=m.id)
            LEFT JOIN options o ON (o.deal_id=d.id)
        -- where d.id=151
        GROUP BY d.deal_uuid
    """
cursor.execute(sql_query)

rows = cursor.fetchall()
with tqdm(total=len(rows)) as pbar:
    for row in rows:

        # index when deal have category only
        if row[2]:
            try:

                #print (f"Row 1: {row[1]}")
                category = get_taxonomy_path(row[2], cursor)
                document = row[1].encode().decode('unicode_escape')
                document = remove_prepositions_and_conjunctions(document + '. ' + category.strip() + '.')
                embedding = model.encode(document).tolist()


                body = {
                    #'id': row[0],
                    'deal_uuid' : row[0],
                    'document': document,
                    'category' : category,
                    'embedding' : embedding,
                    'timestamp': int(datetime.now(timezone.utc).timestamp())
                }    
                #print (body)
                #break

                client.collections['deals'].documents.upsert(body)
            except Exception as e:
                print (f"Exception.: {e} Row 1 {row[1]}.")
                print (f"Error in document id.: {row[0]}.")
                print (f"Body: {body}.")
                errors = errors + 1
                raise
            
            counter = counter + 1
        counterTotal = counterTotal + 1
        pbar.update(1)
    
    conn.close()
    print (f"Errors: {errors}.")
    print (f"Total feeds: {counterTotal}.")
    print (f"Indexed feeds: {counter}.")
    print (f"Deals without category: {counterTotal - counter}.")

  0%|          | 0/105789 [00:00<?, ?it/s]

/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_84671/117606957.py:35: DeprecationWarning: invalid escape sequence '\S'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_84671/117606957.py:35: DeprecationWarning: invalid escape sequence '\A'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_84671/117606957.py:35: DeprecationWarning: invalid escape sequence '\o'
  document = row[1].encode().decode('unicode_escape')
/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_84671/117606957.py:35: DeprecationWarning: invalid escape sequence '\ '
  document = row[1].encode().decode('unicode_escape')


Errors: 0.
Total feeds: 105789.
Indexed feeds: 93044.
Deals without category: 12745.


In [9]:
search_query = 'massage'
start_time = time.time()


response = client.collections['deals'].documents.search({
    'q': search_query,
    'per_page' : 20,
    'query_by': 'document',
    'exclude_fields': 'embedding'
})

elapsed_time = time.time() - start_time
rows = []

for result in response['hits']:
    doc = result['document']
    #print (hit)
    
    rows.append({
        'deal_uuid': doc.get('deal_uuid'),
        'document': doc.get('document'),
        'category': doc.get('category'),
    })

df = pd.DataFrame(rows)

# Výpis statistik
display(HTML(f"<strong>Retrieval: {len(df)} deals. In {elapsed_time:.6f} s.</strong>"))

# Výpis první 1000 řádků jako HTML tabulka
display(HTML(df.head(1000).to_html(escape=False)))

,deal_uuid,document,category
0,fffb2773-8ed3-4cdc-a600-375e4e70146b,discover viva face body rejuvenating facial options like deep sea skin rejuv omega hydro with up to % . one -minute classic facial with -minute massage. luxury -karat gold facialomega hydro one-time facialone -minute classic facial with -minute massage. vip face body. nearby / beauty & spas / face & skin care.,Nearby / Beauty & Spas / Face & Skin Care
1,ffecb490-7d95-4647-afeb-a8143d4afaec,indulge deep tissue swedish body massages at dream kurves with up to % revitalizing experience. deep tissue body massage. swedish body massagedeep tissue body massage. dream kurves. nearby / beauty & spas.,Nearby / Beauty & Spas
2,ffebef95-2f96-46e1-8b1c-2060f0072a23,experience ityra body therapies: foot massages reflexology & more up to % . one -minute foot massage with foot scrub. one -minute foot massage with foot scrub reflexologyone -minute foot massage with foot scrub. ityra body therapies llc. nearby / beauty & spas / massage.,Nearby / Beauty & Spas / Massage
3,ffe9c24a-1cb4-4ea9-8ef1-7bbf77e1138c,enjoy specialty massage at soothing touch with foot face add-ons up to % . one -minute specialty massage with add foot facial treatment. one -minute specialty massage with add face cleanse treatmentone -minute specialty massage with add foot facial treatment. soothing touch. nearby / beauty & spas.,Nearby / Beauty & Spas
4,ffdbfd25-d824-4f28-9ef4-7b60d21c3d57,treat yourself at massage advanced spa: couples acupressure relaxing reflexology & foot scrub!up to % . one mins relax reflexology + foot scurb. one mins couple therapeutic acupressure + reflexology + foot scurbone mins relax reflexology + foot scurb. massage advanced spa. nearby / beauty & spas / massage.,Nearby / Beauty & Spas / Massage
5,ffd0c8b3-04ac-4265-b1c2-12c2eba690e3,indulge swedish massage experience with shin chen at connected health cedar park offering up to % .. two -minute massages at connected health cedar park. one -minute massage at connected health cedar parktwo -minute massages at connected health cedar park. connected health. nearby / beauty & spas / massage.,Nearby / Beauty & Spas / Massage
6,ffb53df7-38bc-4350-ab49-211526038035,escape to pure bliss: choice one -minute massage one -minute deep tissue swedish massage one massage one facial two people. choice -minute swedish massage deep tissue massage facial. one massage one facial two peopleone -minute couples massage valid monday-thursdayone -minute deep tissue swedish massagechoice -minute swedish massage deep tissue massage facial. elements massage - frisco tx. nearby / beauty & spas / massage.,Nearby / Beauty & Spas / Massage
7,ffb2a1bb-45f2-4463-804d-7cd76b497fb0,achieve smooth skin with radiant skin massage studio waxing options up to % . one women brazilian wax. three women brazilian waxesone women brazilian wax. radiant skin massage studio. nearby / beauty & spas / hair removal.,Nearby / Beauty & Spas / Hair Removal
8,ff91d900-bb3a-439f-b615-079162082e2d,discover natural therapeutics -minute massages offering up to % truly rejuvenating experience. one -minute neuromuscular massage. one -minute therapeutic massageone -minute neuromuscular massage. natural therapeutics. nearby / beauty & spas.,Nearby / Beauty & Spas
9,ff8dab75-aeda-46f8-aaf6-b630ad40084c,choice -minute -minute couples massage with choice elevation add- at massage heights % . -minute massage with choice one elevation add-. -minute massage with choice one elevation add--minute couples massage with choice one elevation add- each-minute massage with choice one elevation add-. massage heights buckhead. nearby / beauty & spas / massage.,Nearby / Beauty & Spas / Massage


In [11]:
search_query = 'gordon ramsey'



query_vector = model.encode(search_query).tolist()
start_time = time.time()

# Příklad listu requestů
search_requests = {
    'searches': [
    {
        'collection': 'deals',
        'q': '*',
        'exclude_fields': 'embedding',
        'vector_query': 'embedding:(%s, k:100)' % query_vector,
    }
]
}

common_search_params = {
    'page': 1,
    'per_page': 100
}
response = client.multi_search.perform(search_requests, common_search_params)

end_time = time.time()
elapsed_time = end_time - start_time

rows = []

for result in response['results']:
    hits = result.get('hits', [])
    for hit in hits:
        doc = hit['document']
        #print (hit)
        # Přidejte atributy do seznamu řádků (např. score, deal_uuid, document, category)
        rows.append({
            'deal_uuid': doc.get('deal_uuid'),
            'document': doc.get('document'),
            'category': doc.get('category'),
            'score': hit.get('vector_distance')  # nebo jiný atribut relevance, pokud existuje
        })

df = pd.DataFrame(rows)

# Výpis statistik
display(HTML(f"<strong>Retrieval: {len(df)} deals. In {elapsed_time:.6f} s.</strong>"))

# Výpis první 1000 řádků jako HTML tabulka
display(HTML(df.head(1000).to_html(escape=False)))

,deal_uuid,document,category,score
0,8e5baade-5a75-43eb-aa2b-54e62128f4f1,tony boselli signed bsn orange endzone football pylon /hof - schwartz coa. tony boselli signed bsn orange endzone football pylon /hof - schwartz coa. tony boselli signed bsn orange endzone football pylon /hof - schwartz coa. schwartz sports memorabilia inc.. goods / sports & outdoors / fan shop.,Goods / Sports & Outdoors / Fan Shop,0.680037
1,7064e460-51e7-48dc-ae31-6ac281783464,eddie palmieri at lehman center april at ... // at ..: eddie palmieri. // at ..: eddie palmieri. adalberto santiago lehman center bronx new york . nearby / things to do / fun & leisure.,Nearby / Things To Do / Fun & Leisure,0.723170
2,433e9398-fc1e-41ae-af2f-cd88a3335d1d,steve deberg signed bsn orange football endzone pylon / yds tds. steve deberg signed bsn orange football endzone pylon / yds tds. steve deberg signed bsn orange football endzone pylon / yds tds. schwartz sports memorabilia inc.. goods / sports & outdoors / fan shop.,Goods / Sports & Outdoors / Fan Shop,0.726353
3,208c7c9c-f9c6-492e-a704-3e06db52dc2b,who bad - ultimate michael jackson experience february at pm. // at pm: one ticket. // at pm: one ticket. soul jam. nearby / things to do / fun & leisure.,Nearby / Things To Do / Fun & Leisure,0.731352
4,a95352e4-f8dc-4de8-a50d-cd1e49a67fca,mio marino mens shoes - fashion casual oxford shoes - round toe dress claviko. mio marino mens shoes - fashion casual oxford shoes - round toe dress claviko medium chocolate. mio marino mens shoes - fashion casual oxford shoes - round toe dress claviko black mediummio marino mens shoes - fashion casual oxford shoes - round toe dress claviko . black mediummio marino mens shoes - fashion casual oxford shoes - round toe dress claviko medium chocolate. marino avenue inc.. goods / men fashion / men shoes.,Goods / Men's Fashion / Men's Shoes,0.732370
5,6b7c94f1-fbce-4931-89cb-ab594ada54ae,lenny moore signed colts logo wilson brown full size football /hof. lenny moore signed colts logo wilson brown full size football /hof. lenny moore signed colts logo wilson brown full size football /hof. schwartz sports memorabilia inc.. goods / sports & outdoors / fan shop.,Goods / Sports & Outdoors / Fan Shop,0.733154
6,d4f25c4f-dca8-4bdd-a20d-1cb9325f033f,tony boselli signed jaguars riddell / speed replica helmet /hof white ink. tony boselli signed jaguars riddell / speed replica helmet /hof white ink. tony boselli signed jaguars riddell / speed replica helmet /hof white ink. schwartz sports memorabilia inc.. goods / sports & outdoors / fan shop.,Goods / Sports & Outdoors / Fan Shop,0.736166
7,3188550a-df82-4cd9-b9e3-2f900148e7b4,legends laughter / sommore lavell crawford don dc curry guy torry finesse mitchell & leon rogers / at pm. // at ..: one ticket. // at ..: one ticket. legends laughter . nearby / things to do / nightlife.,Nearby / Things To Do / Nightlife,0.736826
8,79b5428a-f48e-4e84-8e8f-4a45fd287caf,mio marino men casual italian driving loafers / designer gift bag. mio marino men casual italian driving loafers / designer gift bag . medium suave leather penny loafer - tan. mio marino men casual italian driving loafers / designer gift bag medium suave leather penny loafer - tanmio marino men casual italian driving loafers / designer gift bag . medium cultured pebble leather loafer - tanmio marino men casual italian driving loafers / designer gift bag medium suave leather penny loafer - tanmio marino men casual italian driving loafers / designer gift bag . medium suave leather penny loafer - tanmio marino men casual italian driving loafers / designer gift bag . medium suave leather penny loafer - tan. marino avenue inc.. goods / men fashion / men shoes.,Goods / Men's Fashion / Men's Shoes,0.737256
9,47671e83-b684-4c93-b1d0-e4a151e8b9e9,tony boselli signed teal custom football jersey /hof - schwartz coa. tony boselli signed teal custom football jersey /hof - schwartz coa. tony boselli signed teal custom football jersey /hof - schwartz coa. sch